
# Understanding Naive Bayes (step by step with numbers)

Let’s go through a **simple example** by hand before touching any code.  
We’ll predict whether to **Play Tennis** (`Yes` or `No`) based on the **Weather** (`Sunny`, `Overcast`, `Rain`).

---

### The Data

| Weather   | Play |
|------------|------|
| Sunny      | No   |
| Sunny      | No   |
| Overcast   | Yes  |
| Rain       | Yes  |
| Rain       | No   |
| Overcast   | Yes  |
| Sunny      | Yes  |
| Rain       | Yes  |

Total samples: **8**

---

## Step 1: Compute Priors — P(Play=Yes) and P(Play=No)

We simply count how many times each class appears:

- `Play = Yes` → 5 times  
- `Play = No` → 3 times  
- Total = 8  

So:

\[
P(Play=Yes) = 5/8 = 0.625
\]
\[
P(Play=No) = 3/8 = 0.375
\]

These are our **priors** — how likely each class is before seeing any feature.

---

## Step 2: Compute Likelihoods — P(Weather | Play)

We look at how often each type of weather appears **within each class**.

| Weather   | Count when Play=Yes | Count when Play=No |
|------------|--------------------|--------------------|
| Sunny      | 1                  | 2                  |
| Overcast   | 2                  | 0                  |
| Rain       | 2                  | 1                  |
| **Total**  | 5                  | 3                  |

---

### Step 2a: Apply Laplace Smoothing (α = 1)

Without smoothing, if a combination never happened (e.g., `Overcast` with `Play=No`), its probability would be zero — which would “kill” the product later.  
So we add 1 to each count, and adjust the denominator accordingly.

Each class has 3 possible weather types (`Sunny`, `Overcast`, `Rain`), so:

$[ P(Weather=w | Play=c) = \frac{count(w,c) + 1}{total_c + 3} ]$

Now compute:

For **Play=Yes** (total=5):
- P(Sunny | Yes) = (1 + 1) / (5 + 3) = 2/8 = **0.25**  
- P(Overcast | Yes) = (2 + 1) / (5 + 3) = 3/8 = **0.375**  
- P(Rain | Yes) = (2 + 1) / (5 + 3) = 3/8 = **0.375**

For **Play=No** (total=3):
- P(Sunny | No) = (2 + 1) / (3 + 3) = 3/6 = **0.5**  
- P(Overcast | No) = (0 + 1) / (3 + 3) = 1/6 ≈ **0.16**  
- P(Rain | No) = (2 + 1) / (3 + 3) = 3/6 = **0.5**

---

## Step 3: Predict for a new day

Let’s say the weather today is **Overcast**.  
We want to find which class has the higher posterior probability.

### Step 3a: Compute the "score" for each class

We use Bayes’ rule, ignoring the common denominator:

\[ P(Play=c | Weather=Overcast) ∝ P(Play=c) × P(Weather=Overcast | Play=c) \]

Compute both sides:

- For `Play=Yes`:  
  0.625 × 0.375 = **0.234**
- For `Play=No`:  
  0.375 × 0.16 = **0.06**

Normalize:

Sum = 0.234 + 0.06 = 0.294  
So:

\[ P(Yes | Overcast) = 0.234 / 0.294 = 0.79 \]
\[ P(No | Overcast) = 0.06 / 0.294 = 0.21 \]

✅ Therefore, we predict **Play = Yes** (because 0.79 > 0.21)

---

## Step 4: Why it’s called “Naive”

Naive Bayes assumes that all features (Weather, Wind, Temperature, etc.) are **independent** given the class.  
That’s almost never perfectly true — but surprisingly, it still works very well in practice.

---

## Step 5: Key takeaways

- **Priors:** How common each class is.  
- **Likelihoods:** How common each feature value is within each class.  
- **Posterior:** Multiply priors × likelihoods, then normalize.  
- **Laplace smoothing:** Prevents zero probabilities.  
- **Prediction:** Choose the class with the higher posterior.

Next, we’ll use the **same idea in Python** to calculate everything automatically — then compare to `scikit-learn`.


# Naive Bayes — Beginners' Hands‑On (from scratch first)

**You’ll learn:** compute priors → likelihoods (Laplace smoothing) → posteriors (via log-sums).  
**Flow:** tiny categorical example → tiny text classifier from scratch → quick exercises → sklearn at the end.

## 0) Imports (only Python stdlib; sklearn used *once* at the end)

In [ ]:
import math
from collections import Counter, defaultdict

## 1) Categorical toy example — weather → play

We predict `Play ∈ {Yes, No}` from two features: `Outlook ∈ {Sunny, Overcast, Rain}` and `Windy ∈ {True, False}`.
We'll **build all probabilities by counting**.

In [ ]:
# Tiny dataset
data = [
    ("Sunny",   False, "No"),
    ("Sunny",   True,  "No"),
    ("Overcast",False, "Yes"),
    ("Rain",    False, "Yes"),
    ("Rain",    True,  "No"),
    ("Overcast",True,  "Yes"),
    ("Sunny",   False, "Yes"),
    ("Rain",    False, "Yes"),
]

# 1) Priors P(C)
N = len(data)
classes = [play for outlook, windy, play in data]
class_counts = Counter(classes)
priors = {c: class_counts[c]/N for c in class_counts}

print("Priors P(C):", priors)

# 2) Likelihood tables P(feature=value | C) with Laplace smoothing
alpha = 1.0
outlook_vals = {"Sunny","Overcast","Rain"}
windy_vals = {True, False}

# counts per class
cnt_outlook = {c: Counter() for c in class_counts}
cnt_windy = {c: Counter() for c in class_counts}
for outlook, windy, c in data:
    cnt_outlook[c][outlook] += 1
    cnt_windy[c][windy] += 1

def smoothed_prob(counts, value, vocab_size, alpha=1.0):
    num = counts.get(value, 0) + alpha
    den = sum(counts.values()) + alpha * vocab_size
    return num / den

def posterior(outlook, windy, alpha=1.0):
    scores = {}
    for c in class_counts:
        scores[c] = priors[c]
        scores[c] *= smoothed_prob(cnt_outlook[c], outlook, len(outlook_vals), alpha)
        scores[c] *= smoothed_prob(cnt_windy[c], windy, len(windy_vals), alpha)
    Z = sum(scores.values())
    return {c: s/Z for c,s in scores.items()}

print("\nPosterior for (Sunny, False):", posterior("Sunny", False, alpha))
print("Posterior for (Rain, True):", posterior("Rain", True, alpha))

Priors P(C): {'No': 0.375, 'Yes': 0.625}

Posterior for (Sunny, False): {'No': 0.40191387559808617, 'Yes': 0.5980861244019139}
Posterior for (Rain, True): {'No': 0.5283018867924528, 'Yes': 0.4716981132075472}


### ✍️ Exercise A (2–3 min)
1. Change `alpha` to `0.01` and `2.0`. How does `posterior("Rain", True)` shift?
2. Compute by hand (paper) which class is more likely for `(Overcast, False)` and then verify by running:

In [ ]:
posterior("Overcast", False, alpha)  # run this to check your calculation

{'No': 0.12993039443155452, 'Yes': 0.8700696055684455}

## 2) From-scratch Multinomial Naive Bayes for tiny text (no libraries)

**Idea:** count words per class → smooth → multiply per-word likelihoods with the prior (add logs). Out‑of‑vocabulary words are ignored.

In [ ]:
X = [
    "win a free trip now",
    "limited time offer just for you",
    "call mom after class",
    "see you at lunch",
    "cheap meds available online",
    "meet me at the station",
]
y = ["spam","spam","ham","ham","spam","ham"]

def tokenize(s): return s.lower().split()

# Build vocab & counts
V = set()
class_doc_counts = Counter(y)
word_counts = {c: Counter() for c in class_doc_counts}
total_words = {c: 0 for c in class_doc_counts}
for text, c in zip(X, y):
    for w in tokenize(text):
        V.add(w); word_counts[c][w] += 1; total_words[c] += 1
V = sorted(V)

print(word_counts)
print(total_words)
print(V)
print("-"*200)


alpha = 1.0
priors_text = {c: class_doc_counts[c]/len(y) for c in class_doc_counts}

def p_w_given_c(w, c, alpha=1.0):
    return (word_counts[c][w] + alpha) / (total_words[c] + alpha*len(V))

def predict_text_proba(msg, alpha=1.0):
    toks = [w for w in tokenize(msg) if w in V]
    scores = {}
    for c in class_doc_counts:
        scores[c] = priors_text[c]
        for w in toks: scores[c] *= p_w_given_c(w, c, alpha)
    Z = sum(scores.values())
    return {c: s/Z for c,s in scores.items()}

tests = ["free offer for you", "see you at the station", "winner claim your free trip"]
for t in tests:
    post = predict_text_proba(t, alpha=1.0)
    print(f"{t!r} → {post}  predicted={max(post, key=post.get)}")

{'spam': Counter({'win': 1, 'a': 1, 'free': 1, 'trip': 1, 'now': 1, 'limited': 1, 'time': 1, 'offer': 1, 'just': 1, 'for': 1, 'you': 1, 'cheap': 1, 'meds': 1, 'available': 1, 'online': 1}), 'ham': Counter({'at': 2, 'call': 1, 'mom': 1, 'after': 1, 'class': 1, 'see': 1, 'you': 1, 'lunch': 1, 'meet': 1, 'me': 1, 'the': 1, 'station': 1})}
{'spam': 15, 'ham': 13}
['a', 'after', 'at', 'available', 'call', 'cheap', 'class', 'for', 'free', 'just', 'limited', 'lunch', 'me', 'meds', 'meet', 'mom', 'now', 'offer', 'online', 'see', 'station', 'the', 'time', 'trip', 'win', 'you']
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
'free offer for you' → {'spam': 0.8675421778610884, 'ham': 0.1324578221389116}  predicted=spam
'see you at the station' → {'spam': 0.031428537985115385, 'ham': 0.9685714620148846}  predicted=ham
'winner claim your free trip'

### ✍️ Exercise B (3–4 min)
- Add two more **spam** examples that contain the word `"winner"`. Re-run the cell and observe how the probability for `"winner claim your free trip"` changes. Explain the change using counts.
- Try `alpha=0.2` and `alpha=2.0`. What happens when smoothing is too small or too large?

## 3) (Optional) Wrap it as a tiny class

In [ ]:
class MiniMultinomialNB:
    def __init__(self, alpha=1.0):
        self.alpha = alpha
    def _tok(self, s): return s.lower().split()
    def fit(self, X, y):
        self.classes_ = sorted(set(y))
        self.class_counts_ = Counter(y)
        self.N_ = len(y)
        self.priors_ = {c: self.class_counts_[c]/self.N_ for c in self.classes_}
        self.word_counts_ = {c: Counter() for c in self.classes_}
        self.total_words_ = {c: 0 for c in self.classes_}
        V = set()
        for text, c in zip(X, y):
            for w in self._tok(text):
                V.add(w); self.word_counts_[c][w] += 1; self.total_words_[c] += 1
        self.vocab_ = sorted(V); self.V_ = len(self.vocab_); return self
    def _p(self, w, c):
        return (self.word_counts_[c][w] + self.alpha) / (self.total_words_[c] + self.alpha*self.V_)
    def predict_proba(self, X):
        outs = []
        for text in X:
            toks = [w for w in self._tok(text) if w in self.vocab_]
            scores = {}
            for c in self.classes_:
                scores[c] = self.priors_[c];
                for w in toks: scores[c] *= self._p(w,c)
            Z = sum(scores.values()); outs.append({c: s/Z for c,s in scores.items()})
        return outs
    def predict(self, X): return [max(p, key=p.get) for p in self.predict_proba(X)]

mini = MiniMultinomialNB(alpha=1.0).fit(X, y)
mini.predict(["claim your free trip now"]), mini.predict_proba(["claim your free trip now"])[0]

(['spam'], {'ham': 0.12681586757759813, 'spam': 0.873184132422402})

## 4) Finally: how the library does the same thing (one cell)

Once you understand the counts, the library wraps it up neatly. (Run this if you have scikit‑learn installed.)

In [ ]:
try:
    from sklearn.feature_extraction.text import CountVectorizer
    from sklearn.naive_bayes import MultinomialNB
    from sklearn.pipeline import make_pipeline
    sk = make_pipeline(CountVectorizer(), MultinomialNB(alpha=1.0))
    sk.fit(X, y)
    msg = ["claim your free trip now"]
    print("Our class  :", mini.predict(msg)[0], mini.predict_proba(msg)[0])
    print("Sklearn    :", sk.predict(msg)[0], sk.predict_proba(msg)[0])
except Exception as e:
    print("Sklearn not available here — run locally to compare.")
    print("Error:", e)

Our class  : spam {'ham': 0.12681586757759802, 'spam': 0.8731841324224019}
Sklearn    : spam [0.11904394 0.88095606]


## 5) Quick recap
- **Train = count things.** Priors are class frequencies. Likelihoods are per‑class feature frequencies (smoothed).
- **Predict = multiply** per‑feature likelihoods × prior; normalize to get probabilities.
- **Smoothing matters.** α too small → zero-ish probabilities; α too large → everything looks similar.
- **Library = same math, packaged.**